# Schedule building with `ScheduleBuilder` (reference)

**Deep dive:** fluent configuration of payment/roll schedules — frequency, stubs, CDS IMM mode, end-of-month rolls, and `ScheduleErrorPolicy`.


## Concept

- **`Schedule.builder(start, end)`** accumulates **mutable** state: set **`frequency`**, optional **`stub_rule`**, optional **`adjust_with`**, optional **`end_of_month`**, optional **`cds_imm()` / `imm()`**, optional **`error_policy`**, then **`build()`**.
- **Stubs** control where an irregular first/last period sits: **short** vs **long**, **front** vs **back**.
- **End-of-month**: when the start is month-end, rolls track month-ends (February, holidays, etc.).
- **CDS IMM** / **IMM** modes align rolls to market-standard IMM patterns.
- **`ScheduleErrorPolicy`** chooses between **strict** failure, **warnings** when a calendar id is missing, or **graceful empty** schedules.


## API walkthrough

### Basics: `frequency` and `build()`

Frequency accepts **`Tenor.parse`-compatible strings** (e.g. `"6M"`, `"3M"`) or a `Tenor` object. Default stub is **none** — periods divide evenly when possible.


In [ ]:
from datetime import date

from finstack_quant.core.dates import Schedule, StubKind, BusinessDayConvention
print("StubKind.NONE:", StubKind.NONE)
b = Schedule.builder(date(2024, 1, 15), date(2025, 1, 15))
print("Fresh builder:", b)
b.frequency("6M")
b.adjust_with(BusinessDayConvention.MODIFIED_FOLLOWING, "usny")
sched = b.build()
print("len:", len(sched))
print("dates:", [str(d) for d in sched.dates])


### Stub rules (`StubKind`)

**`NONE`**, **`SHORT_FRONT`**, **`SHORT_BACK`**, **`LONG_FRONT`**, **`LONG_BACK`** — use `from_name` for string parsing.


In [ ]:
from datetime import date

from finstack_quant.core.dates import Schedule, StubKind, BusinessDayConvention
start, end = date(2024, 3, 20), date(2027, 5, 20)
for stub in [StubKind.NONE, StubKind.SHORT_BACK, StubKind.SHORT_FRONT, StubKind.LONG_BACK, StubKind.LONG_FRONT]:
    b = Schedule.builder(start, end)
    b.frequency("3M")
    b.stub_rule(stub)
    b.adjust_with(BusinessDayConvention.MODIFIED_FOLLOWING, "target2")
    try:
        s = b.build()
    except ValueError as exc:
        print(f"{stub!s:16s} error={exc}")
        continue
    print(f"{stub!s:16s} n={len(s.dates):2d} first={s.dates[1]} last={s.dates[-1]}")
print("from_name('short_front') == SHORT_FRONT:", StubKind.from_name("short_front") == StubKind.SHORT_FRONT)


### CDS IMM dates and generic IMM rolls

- **`cds_imm()`** — standard CDS roll pattern (quarterly IMM-style anchors).
- **`imm()`** — futures-style IMM schedule (distinct from plain quarterly).


In [ ]:
from datetime import date

from finstack_quant.core.dates import Schedule, BusinessDayConvention
b_cds = Schedule.builder(date(2024, 3, 20), date(2029, 3, 20))
b_cds.cds_imm()
b_cds.adjust_with(BusinessDayConvention.FOLLOWING, "usny")
cds = b_cds.build()
print("CDS IMM (first 6):", [str(d) for d in cds.dates[:6]])

b_imm = Schedule.builder(date(2024, 1, 15), date(2025, 1, 15))
b_imm.imm()
b_imm.adjust_with(BusinessDayConvention.FOLLOWING, "usny")
imm = b_imm.build()
print("IMM mode dates:", [str(d) for d in imm.dates])


### End-of-month convention

Enable with **`end_of_month(True)`** so monthly rolls respect month-end alignment (e.g. Jan-31 → Feb-29 in leap years).


In [ ]:
from datetime import date

from finstack_quant.core.dates import Schedule, BusinessDayConvention
b = Schedule.builder(date(2024, 1, 31), date(2025, 1, 31))
b.frequency("1M")
b.end_of_month(True)
b.adjust_with(BusinessDayConvention.MODIFIED_FOLLOWING, "usny")
s = b.build()
print("EOM monthly (sample):", [str(d) for d in s.dates[:4]], "...", [str(d) for d in s.dates[-3:]])


### `ScheduleErrorPolicy`

- **`STRICT`** — missing calendar or invalid spec **raises**.
- **`MISSING_CALENDAR_WARNING`** — warn and **skip** adjustment if the calendar id is unknown.
- **`GRACEFUL_EMPTY`** — return an **empty** `Schedule` and record warnings / fallback flags.


In [ ]:
from datetime import date

from finstack_quant.core.dates import Schedule, ScheduleErrorPolicy, BusinessDayConvention
bad = "definitely_not_a_calendar"

for label, policy in (
    ("GRACEFUL_EMPTY", ScheduleErrorPolicy.GRACEFUL_EMPTY),
    ("MISSING_CALENDAR_WARNING", ScheduleErrorPolicy.MISSING_CALENDAR_WARNING),
):
    b = Schedule.builder(date(2024, 1, 15), date(2025, 1, 15))
    b.frequency("6M")
    b.adjust_with(BusinessDayConvention.FOLLOWING, bad)
    b.error_policy(policy)
    try:
        b.build()
    except ValueError as exc:
        print(f"{label} failed closed:", str(exc).split(":", 1)[0])


## Practical example

Three real-world patterns:

1. **USD semi-annual corporate-style bond** — 6M frequency, **Modified Following** on **USNY**.
2. **EUR quarterly swap** — 3M, **short back stub**, **TARGET2**.
3. **CDS standard rolls** — `cds_imm()` with **USNY** adjustment.


In [ ]:
from datetime import date

from finstack_quant.core.dates import Schedule, StubKind, BusinessDayConvention
issue, maturity = date(2024, 1, 15), date(2029, 1, 15)
b_bond = Schedule.builder(issue, maturity)
b_bond.frequency("6M")
b_bond.adjust_with(BusinessDayConvention.MODIFIED_FOLLOWING, "usny")
bond = b_bond.build()
print("1) Semi-annual bond (USNY):", len(bond.dates), "dates")
print("   head:", [str(d) for d in bond.dates[:3]])
print("   tail:", [str(d) for d in bond.dates[-2:]])

b_swap = Schedule.builder(date(2024, 3, 20), date(2027, 3, 20))
b_swap.frequency("3M")
b_swap.stub_rule(StubKind.SHORT_BACK)
b_swap.adjust_with(BusinessDayConvention.MODIFIED_FOLLOWING, "target2")
swap = b_swap.build()
print("2) Quarterly swap (T2, short back stub):", len(swap.dates), "dates")
print("   rolls:", [str(d) for d in swap.dates[1:5]], "...")

b_cds = Schedule.builder(date(2024, 3, 20), date(2029, 3, 20))
b_cds.cds_imm()
b_cds.adjust_with(BusinessDayConvention.FOLLOWING, "usny")
cds = b_cds.build()
print("3) CDS IMM (USNY):", len(cds.dates), "dates")
print("   first rolls:", [str(d) for d in cds.dates[:5]])


## Takeaways

- Configure **`ScheduleBuilder`** with **imperative** calls (not a fluent returning `self`); then **`build()`** once.
- Choose **`stub_rule`** when the tenor does not divide the interval evenly.
- Use **`cds_imm()`** vs plain **`frequency("3M")`** when you need **CDS market roll alignment**; use **`imm()`** for **futures IMM**-style schedules.
- **`end_of_month(True)`** is essential for **bonds** that pay on month-end when the issue is on month-end.
- Pick **`ScheduleErrorPolicy`** for **production** robustness: strict in tests, warnings or graceful in exploratory notebooks.


## Analyst program: accrual conventions and stub boundaries

Day count measures accrual time; business-day adjustment determines actual dates. The short-front and long-back schedules intentionally have irregular periods. Futures IMM dates are third Wednesdays, while standard CDS coupon dates are the twentieth of March, June, September and December.

In [ ]:
from datetime import date
from finstack_quant.core.dates import DayCount, Schedule, StubKind, BusinessDayConvention, adjust, next_imm, next_cds_date, third_friday

start, end = date(2024, 1, 15), date(2024, 7, 15)
fractions = {str(basis): basis.year_fraction(start, end) for basis in (DayCount.ACT_360, DayCount.ACT_365F, DayCount.THIRTY_360, DayCount.ACT_ACT)}
assert abs(DayCount.ACT_360.year_fraction(start, end) - 182 / 360) < 1e-12
assert abs(DayCount.ACT_365F.year_fraction(start, end) - 182 / 365) < 1e-12
assert DayCount.THIRTY_360.year_fraction(start, end) == 0.5
assert abs(DayCount.ACT_ACT.year_fraction(start, end) - 182 / 366) < 1e-12
assert adjust(date(2025, 5, 31), BusinessDayConvention.MODIFIED_FOLLOWING, 'usny') == date(2025, 5, 30)
short_front = Schedule.builder(date(2025, 2, 10), date(2026, 1, 15)).frequency('6M').stub_rule(StubKind.SHORT_FRONT).build()
long_back = Schedule.builder(date(2025, 1, 15), date(2026, 2, 10)).frequency('6M').stub_rule(StubKind.LONG_BACK).build()
assert (short_front.dates[1] - short_front.dates[0]).days < 181
assert (long_back.dates[-1] - long_back.dates[-2]).days > 184
assert next_imm(date(2026, 1, 1)) == date(2026, 3, 18)
assert third_friday(3, 2026) == date(2026, 3, 20)
cds_dates = [next_cds_date(date(2026, month, 1)) for month in (1, 4, 7, 10)]
assert cds_dates == [date(2026, month, 20) for month in (3, 6, 9, 12)]
print(fractions, short_front.dates, long_back.dates, cds_dates)

## Reference periods and calendar adjustments

In [ ]:
from datetime import date
from finstack_quant.core.dates import DayCount,DayCountContext,Thirty360Convention,days_30_360,adjust,BusinessDayConvention
reference_start,reference_end=date(2024,1,15),date(2024,7,15)
partial=date(2024,3,15)
ctx=DayCountContext(frequency='6M',coupon_period=(reference_start,reference_end))
icma=DayCount.ACT_ACT_ISMA.year_fraction(reference_start,partial,ctx)
assert abs(icma-(partial-reference_start).days/(2*(reference_end-reference_start).days))<1e-12
cross_year=DayCount.ACT_ACT.year_fraction(date(2023,12,15),date(2024,1,15))
assert abs(cross_year-(17/365+14/366))<1e-12
variants={str(c):days_30_360(date(2025,2,28),date(2025,3,31),c) for c in (Thirty360Convention.US_SIA,Thirty360Convention.ISDA,Thirty360Convention.EUROPEAN,Thirty360Convention.ITALIAN)}
adjusted={str(c):adjust(date(2025,5,31),c,'usny') for c in (BusinessDayConvention.FOLLOWING,BusinessDayConvention.MODIFIED_FOLLOWING,BusinessDayConvention.PRECEDING)}
assert adjusted[str(BusinessDayConvention.FOLLOWING)]==date(2025,6,2)
print({'ICMA_partial':icma,'ISDA_cross_year':cross_year,'30_360_days':variants,'adjusted_dates':adjusted})


## Coupon dollars under two bases

In [ ]:
from datetime import date
from finstack_quant.core.dates import DayCount
from _shared.analyst_book import instruments
spec=instruments('foundations')['USD-CORP']['instrument']['spec']
principal=float(spec['notional']['amount']);rate=float(spec['cashflow_spec']['fixed']['rate'])
a,b=date(2024,1,15),date(2024,7,15)
actual=principal*rate*DayCount.ACT_360.year_fraction(a,b)
bond_basis=principal*rate*DayCount.THIRTY_360.year_fraction(a,b)
assert abs(actual-bond_basis-principal*rate*(182-180)/360)<1e-8
print({'ACT360_coupon':actual,'30360_coupon':bond_basis,'difference_USD':actual-bond_basis})


## A long final coupon period

In [ ]:
from datetime import date
from finstack_quant.core.dates import Schedule,StubKind,DayCount
schedule=Schedule.builder(date(2025,1,15),date(2026,2,10)).frequency('6M').stub_rule(StubKind.LONG_BACK).build()
assert schedule.dates==[date(2025,1,15),date(2025,7,15),date(2026,2,10)]
accruals=[DayCount.THIRTY_360.year_fraction(a,b) for a,b in zip(schedule.dates[:-1],schedule.dates[1:])]
assert accruals[-1]>accruals[0]
print({'dates':schedule.dates,'30360_year_fractions':accruals})


## Quarterly CDS rolls

In [ ]:
from datetime import date
from finstack_quant.core.dates import next_cds_date,next_imm
rolls=[next_cds_date(date(2026,m,1)) for m in (1,4,7,10)]
assert rolls==[date(2026,m,20) for m in (3,6,9,12)]
assert next_imm(date(2026,1,1)) != rolls[0]
print({'CDS_rolls':rolls,'first_futures_IMM':next_imm(date(2026,1,1))})
